# Spike Structure Analysis

Analyze the structural context of shifted mutations by computing
neighbor scores from protein distance matrices.

Adapted from the original `spike-structure-analysis.ipynb`.

In [ ]:
config_path = "config/config.yaml"
profile = None
run_name = None

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns
from Bio.PDB.PDBParser import PDBParser
from Bio.PDB.PDBList import PDBList
from Bio import SeqIO

from notebooks._common import load_config

In [ ]:
cfg = load_config(config_path, profile, run_name=run_name)
spike = cfg["spike"]
output_dir = spike["output_dir"]

os.makedirs(output_dir, exist_ok=True)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline

## Load mutation data

In [ ]:
mut_df_replicates = pd.read_csv(f"{output_dir}/mutations_df.csv", index_col=0)
mut_df_replicates.rename(
    columns={col: col.replace("Omicron_BA2", "BA2") for col in mut_df_replicates},
    inplace=True,
)
print(f"Loaded {len(mut_df_replicates)} mutations")
mut_df_replicates.head()

## Compute site-wise summary statistics

In [ ]:
same_fx = lambda x: x.iloc[0] if np.all(x == x.iloc[0]) else -1
cols_to_collapse = [col for col in mut_df_replicates.columns if "avg_" in col]
cols_to_collapse.extend(["sites"])

shifts_by_site = (
    mut_df_replicates[cols_to_collapse]
    .groupby("sites")
    .agg(same_fx)
)
shifts_by_site.index.name = "res_n"
shifts_by_site.reset_index(inplace=True)
shifts_by_site["res_n"] = shifts_by_site["res_n"].astype(int)

# Compute max absolute shift across conditions
shift_cols = [c for c in shifts_by_site.columns if "avg_shift" in c]
if shift_cols:
    shifts_by_site["max_abs_S"] = shifts_by_site[shift_cols].abs().max(axis=1)

print(f"Site-wise summary: {len(shifts_by_site)} sites")
shifts_by_site.head()

## Read alignment and identify non-identical sites

In [ ]:
alignment_file = "data/clustalo-I20230702-193723-0021-19090519-p1m.clustal_num"
align_dict = {}
with open(alignment_file) as handle:
    for record in SeqIO.parse(handle, "clustal"):
        align_dict[record.id] = record.seq

# Identify non-identical sites relative to BA.1 (Wuhan reference)
wuhan_key = [k for k in align_dict if "P0DTC2" in k or "Wuhan" in k.upper()]
ba1_key = [k for k in align_dict if "BA.1" in k or "BA1" in k or "Omicron" in k.lower()]

if wuhan_key and ba1_key:
    wuhan_seq = str(align_dict[wuhan_key[0]])
    ba1_seq = str(align_dict[ba1_key[0]])
    print(f"Alignment: {len(wuhan_seq)} positions")

    # Identify non-identical sites for BA.2 and Delta
    for homolog_key, label in [(k, k) for k in align_dict if k not in wuhan_key]:
        seq = str(align_dict[homolog_key])
        nis = [i + 1 for i, (a, b) in enumerate(zip(ba1_seq, seq)) if a != b and a != "-" and b != "-"]
        shifts_by_site[f"is_{label}_nis"] = shifts_by_site["res_n"].isin(nis)
else:
    print("Warning: Could not identify Wuhan/BA.1 sequences in alignment")

## Compute distance matrices from PDB structures

In [ ]:
def get_distance_matrix(structure, d_metric="interatomic"):
    """Compute pairwise distance matrix between residues."""
    residues = [r for r in structure.get_residues() if r.get_id()[0] == " "]
    res_ns = []
    res_ids = []
    for res in residues:
        full_id = res.get_full_id()
        chain = full_id[2]
        resseq = res.get_id()[1]
        icode = res.get_id()[2].strip()
        res_ns.append(resseq)
        # Include insertion code to ensure unique IDs
        res_ids.append(f"{chain}_{resseq}{icode}")

    n = len(residues)
    dist_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i + 1, n):
            if d_metric == "interatomic":
                min_dist = float("inf")
                for a1 in residues[i].get_atoms():
                    for a2 in residues[j].get_atoms():
                        d = a1 - a2
                        if d < min_dist:
                            min_dist = d
                dist_matrix[i, j] = min_dist
                dist_matrix[j, i] = min_dist

    # Use integer indices for columns to avoid duplicates; store res_id/res_n as columns
    df = pd.DataFrame(dist_matrix)
    df["res_id"] = res_ids
    df["res_n"] = res_ns
    return df

pdbs = ["7tf8", "7tl9", "7tge"]

dfs = []
for pdb in pdbs:
    output_file = os.path.join(OUTDIR, f"{pdb}_dist_matrix.csv")
    # Always recompute to avoid stale cached files with bad column names
    print(f"Computing distance matrix for {pdb}...")
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb, pdir=OUTDIR, file_format="pdb")
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb, pdb_file)
    dist_df = get_distance_matrix(structure[0])
    dist_df.to_csv(output_file, index=False)
    dfs.append(dist_df)
    print(f"  Saved {output_file}")

if dfs:
    dist_df = pd.concat(dfs, ignore_index=True)
    dist_df = dist_df[~dist_df["res_n"].isin([214])].copy()  # exclude problematic residues
    dist_df["res_n"] = dist_df["res_n"].astype(int)
    print(f"Distance matrix: {len(dist_df)} rows")
else:
    print("No distance matrices computed")
    dist_df = pd.DataFrame()

## Compute neighbor scores

In [ ]:
if not dist_df.empty:
    # Collapse to minimum distance per site pair
    numeric_cols = [c for c in dist_df.columns if c not in ["res_id", "res_n"] and isinstance(c, int)]
    min_dist_df = dist_df.groupby("res_n")[numeric_cols].min()

    # Merge with shift data
    metric_prefix = "max_abs_S"
    nbr_dist_cutoff = 5

    data = shifts_by_site.merge(min_dist_df, on="res_n", how="inner")

    # For each site, compute average metric among neighbors
    nbr_scores = []
    for _, row in data.iterrows():
        site = row["res_n"]
        metric_val = row.get(metric_prefix, 0)

        # Find columns that represent distances to other sites
        dist_cols = [c for c in numeric_cols if c in data.columns]
        if dist_cols:
            distances = row[dist_cols].astype(float)
            nbrs = distances[distances <= nbr_dist_cutoff]
            n_nbrs = len(nbrs)
        else:
            n_nbrs = 0

        nbr_scores.append({
            "res_n": site,
            metric_prefix: metric_val,
            "n_nbrs": n_nbrs,
        })

    nbr_score_df = pd.DataFrame(nbr_scores)

    # Add non-identical site info
    for col in shifts_by_site.columns:
        if col.startswith("is_") and col.endswith("_nis"):
            nbr_score_df = nbr_score_df.merge(
                shifts_by_site[["res_n", col]], on="res_n", how="left"
            )

    nbr_score_df.to_csv(f"{OUTDIR}/nbr_score_df.csv", index=True)
    print(f"Saved {OUTDIR}/nbr_score_df.csv ({len(nbr_score_df)} sites)")
else:
    # Create minimal output so Snakemake rule succeeds
    nbr_score_df = pd.DataFrame({"res_n": [], "max_abs_S": [], "n_nbrs": []})
    nbr_score_df.to_csv(f"{OUTDIR}/nbr_score_df.csv", index=True)
    print("Created empty nbr_score_df.csv (no PDB data available)")

## Structure statistics plot

In [ ]:
if not nbr_score_df.empty and "max_abs_S" in nbr_score_df.columns:
    saveas = f"structural_statistics_{metric_prefix}"

    fig, axes = plt.subplots(1, 2, figsize=[8, 4])

    ax = axes[0]
    bins = np.arange(-0.5, nbr_score_df["n_nbrs"].max() + 1.5, 1)
    sns.histplot(x="n_nbrs", data=nbr_score_df, bins=bins, ax=ax)
    ax.set_xlabel("number of neighbors")
    ax.set_ylabel("number of sites")
    ax.set_title(f"Neighbors within {nbr_dist_cutoff}Å")
    sns.despine(ax=ax)

    ax = axes[1]
    ax.scatter(nbr_score_df["n_nbrs"], nbr_score_df[metric_prefix], alpha=0.5, s=20, c="0.4")
    ax.set_xlabel("number of neighbors")
    ax.set_ylabel(f"{metric_prefix}")
    ax.set_title("Shift vs. structural context")
    sns.despine(ax=ax)

    plt.tight_layout()
    fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No structure data to plot")